---
# `PDF Loaders in Document Loaders`
---

### PyPDF Loader
- Most Important doc loader
- It helps in reading the PDF Files where every page separate document
- Every page in Pdf -> consider separate documents
- PyPdf loader helps in with text-only content in PDF file.

## Installation 
`
pip install py
`

# `Detailed Notes `

# PDF Loaders in LangChain

> **PDF Loader = A Document Loader that extracts content from PDF files and converts it into LangChain `Document` objects.**

PDF loaders are especially important in **RAG applications**, because many real-world knowledge bases are stored as PDFs:

* Company policies
* Research papers
* Books
* Technical documentation
* Resumes
* Product manuals
* Legal documents
* Course notes
* Reports

---

# 1. Where PDF Loaders Fit in RAG

A PDF loader is part of the **document ingestion/indexing pipeline**.

```text
                    PDF
                     ↓
                PDF Loader
                     ↓
                 Documents
                     ↓
               Text Splitter
                     ↓
                  Chunks
                     ↓
              Embedding Model
                     ↓
                Vector Store
                     ↓
                  Retriever
                     ↓
               Relevant Chunks
                     ↓
                    LLM
                     ↓
                  Answer
```

### Important

The PDF loader **does not**:

* Generate embeddings
* Create vectors
* Search the PDF
* Generate answers
* Act as an LLM

Its primary responsibility is:

> **PDF → LangChain Documents**

---

# 2. What is a PDF Loader?

Suppose we have:

```text
company_policy.pdf
```

The PDF might contain:

```text
Page 1:
Company Introduction

Page 2:
Leave Policy

Page 3:
Work From Home Policy

Page 4:
Employee Benefits
```

A PDF loader reads the PDF and converts its content into `Document` objects.

Conceptually:

```text
company_policy.pdf
        ↓
    PDF Loader
        ↓
┌────────────────────┐
│ Document - Page 1  │
│ Document - Page 2  │
│ Document - Page 3  │
│ Document - Page 4  │
└────────────────────┘
```

---

# 3. What is a `Document`?

A LangChain `Document` generally contains:

```text
Document
├── page_content
└── metadata
```

### `page_content`

The extracted text:

```text
"Employees are entitled to 15 days of annual leave..."
```

### `metadata`

Information about where the content came from:

```python
{
    "source": "company_policy.pdf",
    "page": 2
}
```

The exact metadata fields can vary by loader.

---

# 4. Why Metadata Is Important

Suppose your RAG chatbot answers:

> Employees receive 15 days of annual leave.

You may want to tell the user:

```text
Source: company_policy.pdf
Page: 2
```

That is possible because the document retained metadata.

```text
PDF
 ↓
PDF Loader
 ↓
Document
 ├── page_content
 └── metadata
       ├── source
       └── page
```

### Interview Point

> **Metadata helps with source attribution, filtering, debugging, and citations.**

---

# 5. Popular PDF Loaders in LangChain

LangChain supports different PDF loading approaches because PDFs can be very different.

Common examples include:

* `PyPDFLoader`
* `PyMuPDFLoader`
* `PDFPlumberLoader`
* `UnstructuredPDFLoader`
* Layout-aware loaders for more complex documents

For learning basic PDF RAG, **`PyPDFLoader`** is a good starting point.

---

# 6. `PyPDFLoader`

`PyPDFLoader` is commonly used to extract text from PDFs.

Install the relevant packages:

```bash
pip install langchain-community pypdf
```

Then:

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("company_policy.pdf")

documents = loader.load()
```

Now:

```text
company_policy.pdf
        ↓
PyPDFLoader
        ↓
documents
```

---

# 7. Inspect the Loaded Documents

You can inspect the number of documents:

```python
print(len(documents))
```

If the loader produces one document per page, a 10-page PDF may produce roughly:

```text
10 Documents
```

You can inspect the first document:

```python
print(documents[0].page_content)
```

And its metadata:

```python
print(documents[0].metadata)
```

Conceptually:

```python
{
    "source": "company_policy.pdf",
    "page": 0
}
```

Page numbering may be zero-based depending on the loader.

---

# 8. Important: PDF ≠ One Document

This is an important concept.

A PDF file can contain many pages.

```text
company_policy.pdf
       ↓
Page 1
Page 2
Page 3
Page 4
       ↓
PDF Loader
       ↓
Documents
```

Depending on the loader and configuration, pages may become separate `Document` objects.

This is useful because metadata can tell us which page contains a piece of information.

---

# 9. Example PDF

Imagine `employee_handbook.pdf` contains:

```text
Page 1
Employee Handbook

Page 2
Leave Policy:
Employees receive 15 days of annual leave.

Page 3
Remote Work:
Employees can work remotely twice per week.

Page 4
Benefits:
Employees receive health and retirement benefits.
```

After loading:

```text
Document 1
page_content = "Employee Handbook"
metadata = {"page": 0}

Document 2
page_content = "Leave Policy..."
metadata = {"page": 1}

Document 3
page_content = "Remote Work..."
metadata = {"page": 2}

Document 4
page_content = "Benefits..."
metadata = {"page": 3}
```

---

# 10. PDF Loader + Text Splitter

Usually, we don't directly embed entire PDF pages.

Instead:

```text
PDF
 ↓
PDF Loader
 ↓
Documents
 ↓
Text Splitter
 ↓
Chunks
```

Example:

```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("company_policy.pdf")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")
```

---

# 11. Why Do We Split PDF Content?

Imagine a PDF has:

```text
500 pages
```

Each page could contain a lot of text.

We don't want to blindly send the entire PDF to the LLM.

Instead:

```text
500-page PDF
      ↓
PDF Loader
      ↓
Documents
      ↓
Text Splitter
      ↓
Thousands of chunks
```

Then the retriever can find only the relevant chunks.

---

# 12. PDF Loader vs Text Splitter

Very important interview distinction:

### PDF Loader

```text
PDF
 ↓
Documents
```

### Text Splitter

```text
Document
 ↓
Smaller Chunks
```

Therefore:

> **PDF Loader extracts the content; Text Splitter divides the content.**

---

# 13. PDF Loader vs Embedding

Another common confusion.

### PDF Loader

```text
PDF
 ↓
Text/Documents
```

### Embedding Model

```text
Text
 ↓
Vector
```

For example:

```text
"Employees receive 15 days of annual leave."
```

becomes something conceptually like:

```text
[0.12, -0.45, 0.78, ...]
```

So:

> **Loader extracts text; embedding model converts text into numerical vectors.**

---

# 14. Complete PDF RAG Pipeline

This is the architecture you should memorize:

```text
                     PDF
                      ↓
                 PDF Loader
                      ↓
                  Documents
                      ↓
                Text Splitter
                      ↓
                    Chunks
                      ↓
               Embedding Model
                      ↓
                 Vector Store
                      ↓
                ─────────────
                      ↓
                 User Question
                      ↓
                  Retriever
                      ↓
               Relevant Chunks
                      ↓
                     LLM
                      ↓
                   Answer
```

---

# 15. Mini Project: Chat With Your PDF

Let's build a beginner-friendly project.

## Project

### **Chat with Your PDF**

User uploads:

```text
machine_learning_notes.pdf
```

Then asks:

```text
What is overfitting?
```

The application searches the PDF and generates an answer.

---

# 16. Project Architecture

```text
                 machine_learning_notes.pdf
                              ↓
                         PDF Loader
                              ↓
                          Documents
                              ↓
                        Text Splitter
                              ↓
                            Chunks
                              ↓
                         Embeddings
                              ↓
                        Vector Store
                              ↓
                       ─────────────
                              ↓
                         User Question
                              ↓
                          Retriever
                              ↓
                       Relevant Chunks
                              ↓
                             LLM
                              ↓
                           Answer
```

---

# 17. Project Folder Structure

A simple version:

```text
pdf-rag-chatbot/
│
├── data/
│   └── machine_learning.pdf
│
├── ingest.py
├── chatbot.py
├── requirements.txt
└── README.md
```

---

# 18. Step 1 — Load the PDF

Create:

### `ingest.py`

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "data/machine_learning.pdf"
)

documents = loader.load()

print(f"Number of documents: {len(documents)}")

print("\nFirst page:")
print(documents[0].page_content)

print("\nMetadata:")
print(documents[0].metadata)
```

Output could look like:

```text
Number of documents: 25

First page:
Machine Learning Introduction...

Metadata:
{
    'source': 'data/machine_learning.pdf',
    'page': 0
}
```

---

# 19. Step 2 — Split the PDF

Add:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
```

Now:

```text
PDF
 ↓
25 Documents
 ↓
Text Splitter
 ↓
100+ Chunks
```

The actual number depends on the content and splitter configuration.

---

# 20. Step 3 — Create Embeddings

Now:

```text
Chunks
 ↓
Embedding Model
 ↓
Vectors
```

For example:

```text
Chunk:
"Overfitting occurs when a model performs very well
on training data but poorly on unseen data."

        ↓

Embedding

        ↓

[0.12, -0.32, 0.87, ...]
```

You can use an embedding model from the provider of your choice.

---

# 21. Step 4 — Store in Vector Database

For a learning project, you could use:

```text
Chroma
FAISS
```

For production, you might consider:

```text
Pinecone
Qdrant
Weaviate
```

Architecture:

```text
Chunks
 ↓
Embeddings
 ↓
Vector Database
```

---

# 22. Step 5 — Ask a Question

User:

> "What is overfitting?"

The question goes through:

```text
Question
 ↓
Embedding
 ↓
Vector Search
 ↓
Relevant PDF Chunks
```

Suppose the retriever finds:

```text
Chunk from page 12
```

Then:

```text
Question
+
Relevant Context
 ↓
LLM
 ↓
Answer
```

---

# 23. Example Final Answer

User:

> What is overfitting?

RAG system:

```text
Overfitting occurs when a machine learning model
learns the training data too closely, including noise
and patterns that do not generalize well to unseen data.

Source: machine_learning.pdf
Page: 12
```

This is much better than simply saying:

> "According to my knowledge..."

because the answer is grounded in your document.

---

# 24. Build a Multi-PDF Chatbot

Now make it more interesting.

Suppose:

```text
data/
├── machine_learning.pdf
├── deep_learning.pdf
├── nlp.pdf
├── langchain.pdf
└── rag.pdf
```

Load all PDFs:

```python
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_path in Path("data").glob("*.pdf"):
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()

    documents.extend(docs)

print(f"Total documents: {len(documents)}")
```

Architecture:

```text
           PDF 1 ──┐
           PDF 2 ──┤
           PDF 3 ──┤
           PDF 4 ──┤
           PDF 5 ──┤
                   ↓
              PDF Loaders
                   ↓
               Documents
                   ↓
              Text Splitter
                   ↓
                 Chunks
                   ↓
               Embeddings
                   ↓
              Vector Store
```

---

# 25. Why This Is a Good GenAI Project

This small project teaches many important concepts:

```text
PDF Loading
     ↓
Document Objects
     ↓
Metadata
     ↓
Chunking
     ↓
Embeddings
     ↓
Vector Database
     ↓
Retrieval
     ↓
Prompt
     ↓
LLM
     ↓
RAG
```

It gives you the foundation for larger projects such as:

### 1. Research Paper Assistant

```text
Research Papers
       ↓
PDF RAG
       ↓
Ask Questions
```

### 2. Resume Analyzer

```text
Resume PDFs
       ↓
PDF Loader
       ↓
LLM
       ↓
Skills / Experience Extraction
```

### 3. Legal Document Assistant

```text
Legal PDFs
       ↓
PDF RAG
       ↓
Question Answering
```

### 4. Study Assistant

```text
Course PDFs
       ↓
PDF RAG
       ↓
AI Tutor
```

---

# 26. Important Problem: Scanned PDFs

Not every PDF contains actual text.

There are two broad cases:

### Text-based PDF

```text
PDF
 ↓
Selectable Text
 ↓
PDF Loader
 ↓
Text
```

Easy to process.

### Scanned PDF

```text
PDF
 ↓
Image of Page
 ↓
No actual text layer
```

A normal text-extraction loader may not be able to extract useful text.

You may need:

```text
OCR
 ↓
Extracted Text
 ↓
Documents
```

Possible OCR approaches include tools such as:

* Tesseract
* Cloud OCR services
* Document AI services
* Vision-capable models

### Interview Point

> **A PDF loader that extracts text does not automatically solve OCR for image-only/scanned PDFs.**

---

# 27. Problem: Tables in PDFs

PDFs containing tables can be challenging.

For example:

```text
| Product | Price | Stock |
|---------|-------|-------|
| Laptop  | 80K   | 10    |
```

Simple text extraction may produce poorly structured text.

For complex PDFs containing:

* Tables
* Images
* Columns
* Headers
* Footnotes
* Complex layouts

you may need a more specialized parser/layout-aware extraction strategy.

---

# 28. Choosing a PDF Loader

A simple decision:

```text
Simple text PDF
      ↓
PyPDFLoader
```

```text
Need different PDF parsing behavior
      ↓
Consider PyMuPDF / PDFPlumber
```

```text
Complex document structure
      ↓
Consider layout-aware/document parsing tools
```

```text
Scanned PDF
      ↓
OCR required
```

Don't choose a loader only because it is popular. Choose based on the PDF's actual structure.

---

# 29. Common Mistakes

## Mistake 1 — Sending the entire PDF to the LLM

Bad architecture:

```text
PDF
 ↓
LLM
```

Better:

```text
PDF
 ↓
Loader
 ↓
Split
 ↓
Embed
 ↓
Retrieve Relevant Chunks
 ↓
LLM
```

---

## Mistake 2 — Assuming every PDF is text-readable

A scanned PDF may contain only images.

```text
PDF
 ↓
Image
 ↓
OCR
 ↓
Text
```

---

## Mistake 3 — Ignoring metadata

Always think about:

```text
source
page
document_id
section
```

when building a production RAG system.

---

## Mistake 4 — Using very large chunks

Huge chunks can hurt retrieval quality.

Instead:

```text
Large Document
 ↓
Meaningful Chunks
```

The ideal chunk size depends on your data and retrieval task.

---

# 30. PDF Loader vs Text Loader

| Feature            | `TextLoader`         | PDF Loader             |
| ------------------ | -------------------- | ---------------------- |
| Source             | `.txt`               | `.pdf`                 |
| Parsing complexity | Low                  | Higher                 |
| Page metadata      | Usually not relevant | Often available        |
| Tables             | N/A                  | Can be challenging     |
| Images             | N/A                  | Can require OCR/vision |
| Common RAG use     | Notes/text           | Manuals/reports/books  |

### Simple Rule

```text
.txt → TextLoader

.pdf → PDF Loader
```

---

# 31. Interview Questions

## Beginner

### Q1. What is a PDF Loader in LangChain?

**Answer:**

A PDF loader reads PDF files, extracts their content, and converts it into LangChain `Document` objects containing text and metadata.

---

### Q2. Give an example of a PDF loader.

**Answer:**

`PyPDFLoader` is a commonly used LangChain PDF loader.

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("document.pdf")
documents = loader.load()
```

---

### Q3. What does a PDF loader return?

**Answer:**

It returns `Document` objects containing `page_content` and metadata.

---

### Q4. Does a PDF loader create embeddings?

**Answer:**

No.

```text
PDF Loader → Documents
Embedding Model → Vectors
```

---

# 32. Intermediate Questions

### Q5. Why is metadata important in PDF RAG?

**Answer:**

Metadata can identify the source PDF and page number, allowing source attribution, filtering, debugging, and citations.

---

### Q6. What happens after loading a PDF?

**Answer:**

A typical RAG pipeline is:

```text
PDF
 ↓
Loader
 ↓
Documents
 ↓
Text Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector Store
```

---

### Q7. Can a PDF loader process scanned PDFs?

**Answer:**

Not necessarily. If the PDF contains images instead of a text layer, OCR may be required before or during the document extraction process.

---

# 33. Scenario-Based Questions

### Q8. Your PDF has 500 pages. Would you send all pages to the LLM?

**Answer:**

No. I would load the PDF, split it into chunks, embed and index those chunks, retrieve only the relevant chunks for the user's query, and provide those chunks to the LLM.

---

### Q9. Your RAG system retrieves the right information but cannot show the source page. What would you check?

**Answer:**

I would check whether page/source metadata was preserved during PDF loading, splitting, indexing, and retrieval.

---

### Q10. Your PDF contains many tables and images, but the extracted text is poor. What would you do?

**Answer:**

I would evaluate a more suitable PDF parser or layout-aware document extraction approach. For scanned/image content, I would add OCR or a suitable vision/document-understanding pipeline.

---

# 34. 30-Second Revision

> **PDF Loader converts PDF content into LangChain `Document` objects.**

```text
PDF
 ↓
PDF Loader
 ↓
Documents
├── page_content
└── metadata
```

For RAG:

```text
PDF
 ↓
Loader
 ↓
Documents
 ↓
Text Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector Store
 ↓
Retriever
 ↓
LLM
```

### Remember

> **Loader extracts → Splitter chunks → Embedding vectorizes → Vector Store stores → Retriever retrieves → LLM generates.**

---

# 35. 2-Minute Revision

## PDF Loaders

Used to load PDF files and convert them into LangChain `Document` objects.

### Common Example

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("company.pdf")

documents = loader.load()
```

### Document

```text
Document
├── page_content
└── metadata
```

Example metadata:

```python
{
    "source": "company.pdf",
    "page": 5
}
```

### Complete PDF RAG

```text
               PDF
                ↓
           PDF Loader
                ↓
            Documents
                ↓
          Text Splitter
                ↓
              Chunks
                ↓
          Embedding Model
                ↓
           Vector Store
                ↓
            Retriever
                ↓
         Relevant Chunks
                ↓
               LLM
                ↓
             Answer
```

### Important Differences

| Component         | Job                   |
| ----------------- | --------------------- |
| **PDF Loader**    | Extract PDF content   |
| **Document**      | Store text + metadata |
| **Text Splitter** | Create chunks         |
| **Embedding**     | Create vectors        |
| **Vector Store**  | Store/search vectors  |
| **Retriever**     | Find relevant chunks  |
| **LLM**           | Generate answer       |

### Final Interview Answer

> **PDF loaders in LangChain are components used to extract content from PDF files and convert it into standardized `Document` objects. A document typically contains the extracted text in `page_content` and metadata such as the source and page number. PDF loaders are commonly the first step in a RAG ingestion pipeline, followed by text splitting, embedding, vector storage, retrieval, and finally LLM-based generation. For scanned PDFs or complex layouts, additional OCR or specialized document-parsing techniques may be required.**
